# QLoRA fine-tune Qwen2.5-3B on Kaggle T4

QLoRA uses **Transformers + bitsandbytes 4-bit + LoRA**, not llama.cpp/GGUF.

Before you start:
1. Stop the Gradio cell if it is still running.
2. **Restart the kernel** so GPU memory is empty.
3. Select the **Kaggle remote kernel** again.
4. Run this notebook from the top.

What you get: a small adapter folder at `/kaggle/working/qlora/adapter` (a few hundred MB), not a new full model. At inference you load the 4-bit base + this adapter.

In [2]:
import gc
import os
import shutil
from pathlib import Path

import torch

# --- change these ---
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID = None  # land-law JSONL, not Alpaca
MAX_SAMPLES = None  # use the full converted file
MAX_SEQ_LEN = 768
EPOCHS = 2
LR = 2e-4
LORA_R = 16
LORA_ALPHA = 32
BATCH_SIZE = 1
GRAD_ACCUM = 8

IS_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("./output")
HF_HOME = ROOT / "hf"
OUT_DIR = ROOT / "qlora" / "adapter"
DATA_DIR = Path("/kaggle/input/datasets/mdraiyanbuhiyaloreen/dataset-land-law")
DATA_CANDIDATES = [
    DATA_DIR / "land_law_sft.jsonl",
    DATA_DIR,
    Path("/kaggle/input/dataset-land-law/land_law_sft.jsonl"),
    Path("/kaggle/input/dataset-land-law"),
    ROOT / "land_law_sft.jsonl",
    Path("data/land_law_sft.jsonl"),
]


def resolve_jsonl(paths):
    for p in paths:
        if p.is_file() and p.suffix.lower() in {".jsonl", ".json"}:
            return p
        if p.is_dir():
            hits = sorted(p.rglob("*.jsonl")) + sorted(p.rglob("*.json"))
            if hits:
                return hits[0]
    return None


DATA_PATH = resolve_jsonl(DATA_CANDIDATES)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
HF_HOME.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name in ("model", "tokenizer", "llm", "trainer", "demo"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Re-select the Kaggle kernel after restart.")

print("gpu", torch.cuda.get_device_name(0))
print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("disk free GB", round(shutil.disk_usage(ROOT).free / 1024**3, 1))
print("save adapters to", OUT_DIR)
print("data dir", DATA_DIR, "exists", DATA_DIR.exists())
print("train jsonl", DATA_PATH, "exists", DATA_PATH is not None and DATA_PATH.exists())
if DATA_DIR.exists():
    print("files:", [str(p.relative_to(DATA_DIR)) for p in DATA_DIR.rglob("*") if p.is_file()][:30])

gpu Tesla T4
vram GB 14.6
disk free GB 19.5
save adapters to /kaggle/working/qlora/adapter
data dir /kaggle/input/datasets/mdraiyanbuhiyaloreen/dataset-land-law exists True
train jsonl /kaggle/input/datasets/mdraiyanbuhiyaloreen/dataset-land-law/land_law_sft.jsonl exists True
files: ['land_law_sft.jsonl']


In [3]:
%pip install -q -U peft trl datasets accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 108.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import Dataset, load_dataset
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not hf_token and IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

jsonl_path = DATA_PATH or resolve_jsonl(DATA_CANDIDATES)
if jsonl_path is None:
    searched = "\n".join(str(p) for p in DATA_CANDIDATES)
    raise FileNotFoundError(
        "No .jsonl/.json found. Attach the dataset and check the path. Searched:\n" + searched
    )

train_ds = load_dataset("json", data_files=str(jsonl_path), split="train")
if MAX_SAMPLES:
    train_ds = train_ds.select(range(min(MAX_SAMPLES, len(train_ds))))

print("loaded", jsonl_path)
print(train_ds)
print(train_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

loaded /kaggle/input/datasets/mdraiyanbuhiyaloreen/dataset-land-law/land_law_sft.jsonl
Dataset({
    features: ['messages'],
    num_rows: 1121
})
{'messages': [{'role': 'user', 'content': 'What does Section 3 of the Transfer of Property Act, 1882 say?'}, {'role': 'assistant', 'content': 'Section 3 of the Transfer of Property Act, 1882 provides: In this Act, unless there is something repugnant in the subject or context,- "immoveable property" does not include standing timber, growing crops or grass: "instrument" means a non-testamentary instrument: "attested", in relation to an instrument, means and shall be deemed always to have meant attested by two or more witnesses each of whom has seen the executant sign or affix his mark to the instrument, or has seen some other person sign the instrument in the presence and by the direction of the executant, or has received from the executant a personal acknowledgement of his signature or mark, or of the signature of such other person, and each 

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

compute_dtype = torch.float16  # T4 has no bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print("trainable setup ready", round(model.get_memory_footprint() / 1024**3, 2), "GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable setup ready 2.45 GB


In [6]:
import inspect
import re
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:
    from transformers import TrainingArguments as SFTConfig

print("trl SFTConfig", SFTConfig.__module__)


def filter_kwargs(cls, kwargs):
    params = inspect.signature(cls.__init__).parameters
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
        return dict(kwargs)
    allowed = set(params) - {"self"}
    return {k: v for k, v in kwargs.items() if k in allowed}


def build_config(kwargs):
    pending = dict(kwargs)
    while True:
        try:
            return SFTConfig(**filter_kwargs(SFTConfig, pending))
        except TypeError as e:
            m = re.search(r"unexpected keyword argument '([^']+)'", str(e))
            if not m:
                raise
            pending.pop(m.group(1), None)
            print("dropped unsupported SFTConfig arg:", m.group(1))


def build_trainer(kwargs):
    pending = dict(kwargs)
    while True:
        try:
            return SFTTrainer(**filter_kwargs(SFTTrainer, pending))
        except TypeError as e:
            m = re.search(r"unexpected keyword argument '([^']+)'", str(e))
            if not m:
                raise
            pending.pop(m.group(1), None)
            print("dropped unsupported SFTTrainer arg:", m.group(1))


wanted = dict(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=5,
    save_strategy="epoch",
    fp16=False,  # T4 AMP cannot unscale bfloat16 grads
    bf16=False,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    max_length=MAX_SEQ_LEN,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    eos_token="<|im_end|>",
    loss_type="nll",  # skip TRL chunked-CE patch (crashes on QLoRA partial.forward)
)

args = build_config(wanted)

try:
    trainer = build_trainer(
        dict(
            model=model,
            args=args,
            train_dataset=train_ds,
            peft_config=lora_config,
            processing_class=tokenizer,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            max_length=MAX_SEQ_LEN,
        )
    )
except AttributeError as e:
    print("SFTTrainer failed, using transformers.Trainer fallback:", e)
    from peft import get_peft_model
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

    if not hasattr(model, "peft_config"):
        model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    def tokenize_row(ex):
        text = tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
        toks = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN)
        toks["labels"] = toks["input_ids"].copy()
        return toks

    tokenized = train_ds.map(tokenize_row, remove_columns=train_ds.column_names)
    ta = build_config(
        dict(
            output_dir=str(OUT_DIR),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR,
            logging_steps=5,
            save_strategy="epoch",
            fp16=False,
            bf16=False,
            optim="paged_adamw_8bit",
            lr_scheduler_type="cosine",
            gradient_checkpointing=True,
            report_to="none",
            remove_unused_columns=False,
        )
    )
    # TrainingArguments if SFTConfig leftover fields break Trainer
    try:
        trainer = Trainer(
            model=model,
            args=ta,
            train_dataset=tokenized,
            data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
            processing_class=tokenizer,
        )
    except TypeError:
        trainer = Trainer(
            model=model,
            args=TrainingArguments(
                output_dir=str(OUT_DIR),
                num_train_epochs=EPOCHS,
                per_device_train_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=LR,
                logging_steps=5,
                save_strategy="epoch",
                fp16=False,
                optim="paged_adamw_8bit",
                lr_scheduler_type="cosine",
                gradient_checkpointing=True,
                report_to="none",
            ),
            train_dataset=tokenized,
            data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
            tokenizer=tokenizer,
        )

for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

train_result = trainer.train()
print(train_result.metrics)
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print("adapter saved to", OUT_DIR)

trl SFTConfig trl.trainer.sft_config


Tokenizing train dataset:   0%|          | 0/1121 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1121 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1121 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1121 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.068915
10,1.752079
15,1.543575
20,1.465237
25,1.411099
30,1.392420
35,1.323977
40,1.402030
45,1.337813
50,1.330016


{'train_runtime': 2371.9058, 'train_samples_per_second': 0.945, 'train_steps_per_second': 0.119, 'total_flos': 1.1256681642098688e+16, 'train_loss': 0.9215430431332149, 'entropy': 0.5799237026108636, 'mean_token_accuracy': 0.8637460933791267, 'num_tokens': 668916.0, 'epoch': 2.0}
adapter saved to /kaggle/working/qlora/adapter


In [8]:
model.eval()
model.config.use_cache = True

prompt = "Explain QLoRA in two sentences."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain QLoRA in two sentences.
assistant
QLoRA is an open-source project that provides a framework for building LoRaWAN applications on top of the QoS-Compliant LoRaWAN Network Stack (QLoRA NS). It includes a LoRaWAN application layer stack and a LoRaWAN network management system.


The adapter is only the LoRA weights. To use it later:

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base, "/kaggle/working/qlora/adapter")
```

Download `/kaggle/working/qlora/adapter` from Kaggle output. To train on your own data, put JSONL rows like `{"messages": [{"role":"user","content":"..."}, {"role":"assistant","content":"..."}]}` and set `DATASET_ID = None`, then load that file instead of `LOCAL_EXAMPLES`.